# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all RecordSet @id's and details
print("Available RecordSets:")
for record_set in metadata.record_sets:
    print(f"- RecordSet @id: {record_set['@id']} | name: {record_set.get('name', '')}")
    if 'field' in record_set:
        print("  Fields:")
        for field in record_set['field']:
            print(f"    - Field @id: {field['@id']} | name: {field.get('name', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, get all RecordSet @id's
record_sets = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records from RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"- {record_set_id}: columns: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"- {record_set_id}: [No data loaded]")

# For demonstration, pick the first record set with data (if any)
main_record_set_id = next((r for r in dataframes.keys() if not dataframes[r].empty), None)
if main_record_set_id:
    print(f"Selected main RecordSet: {main_record_set_id}")
    print(f"Available fields: {dataframes[main_record_set_id].columns.tolist()}")
    dataframes[main_record_set_id].head()
else:
    print("No record sets found with data.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# For demo, use the main record set dataframe
if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Attempt to auto-detect numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        numeric_field = numeric_cols[0]  # Use the first numeric column
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean()  # Use mean as threshold for demonstration
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records where {numeric_field} > {threshold:.2f} (mean):")
        print(filtered_df.head())

        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, norm_col]].head())

        # Try grouping by a string/categorical field if available
        group_fields = df.select_dtypes(include=['object']).columns.tolist()
        group_field = group_fields[0] if group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            print(grouped_df.head())
        else:
            print("No categorical field found for grouping.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No data loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_cols:
    df = dataframes[main_record_set_id]
    numeric_field = numeric_cols[0]
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If a group field exists, plot boxplot
    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualizations not available (no numeric or categorical data).")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*This notebook demonstrated how to use the `mlcroissant` library to load, explore, and visualize FAIR mass spectrometry data from the Croissant schema. For your own explorations, use the `@id`s for record sets, fields, and columns as needed, and adjust EDA and visualization code to suit your analysis objectives.*